In [ ]:
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError, PaginationError
from requests.exceptions import HTTPError

import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

API_KEY = "api"
client = EntsoePandasClient(api_key= API_KEY)

In [ ]:
start = pd.Timestamp("2019-01-01", tz="Europe/Brussels")
end   = pd.Timestamp("2020-01-01", tz="Europe/Brussels")

country_code = "NL"

process_types = {
    "A51": "aFRR",
    "A47": "mFRR",
    "A46": "RR"
}
contract_terms = ["A13","A01","A02","A03","A04"]  # hourly,daily,weekly,monthly,yearly

available = []
missing = []
frames = []

#loop process type per contract term
for pt, pname in process_types.items():
    for term in contract_terms:
        try:
            df = client.query_contracted_reserve_prices(
                country_code= country_code,
                start=start, end=end,
                process_type=pt,
                type_marketagreement_type=term,
                psr_type=None
            )
            if not df.empty:
                available.append((pname, pt, term, df.index.min(), df.index.max()))
            else:
                missing.append((pname, pt, term, "empty frame"))
                continue 

            if isinstance(df, pd.Series):
                df = df.to_frame("price_eur_per_mw_term")
            else:
                numcol = "value" if "value" in df.columns else df.columns[0]
                df = df.rename(columns={numcol: "price_eur_per_mw_term"})[["price_eur_per_mw_term"]]
            df = df[~df.index.duplicated(keep="last")]

            df.index.name = "period_start"
            df["product"] = pname
            df["process_type"] = pt
            df["term_code"] = term
            frames.append(df)

            available.append((pname, pt, term, df.index.min(), df.index.max()))
        except NoMatchingDataError:
            missing.append((pname, pt, term, "NoMatchingData"))
        except Exception as e:
            missing.append((pname, pt, term, type(e).__name__))

print("AVAILABLE:")
for row in available:
    print(row)

print("\nMISSING:")
for row in missing:
    print(row)


AVAILABLE:

MISSING:
('aFRR', 'A51', 'A13', 'HTTPError')
('aFRR', 'A51', 'A01', 'HTTPError')
('aFRR', 'A51', 'A02', 'HTTPError')
('aFRR', 'A51', 'A03', 'HTTPError')
('aFRR', 'A51', 'A04', 'HTTPError')
('mFRR', 'A47', 'A13', 'HTTPError')
('mFRR', 'A47', 'A01', 'HTTPError')
('mFRR', 'A47', 'A02', 'HTTPError')
('mFRR', 'A47', 'A03', 'HTTPError')
('mFRR', 'A47', 'A04', 'HTTPError')
('RR', 'A46', 'A13', 'HTTPError')
('RR', 'A46', 'A01', 'HTTPError')
('RR', 'A46', 'A02', 'HTTPError')
('RR', 'A46', 'A03', 'HTTPError')
('RR', 'A46', 'A04', 'HTTPError')
